# RAG (Retrieval-Augmented Generation) from Scratch

Building a complete RAG pipeline using only numpy and scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import re

## 1. Document Collection

In [ ]:
documents = [
    """Machine learning is a subset of artificial intelligence that enables systems to learn from data. 
    Supervised learning uses labeled examples to train models, while unsupervised learning finds patterns 
    in unlabeled data. Deep learning uses neural networks with many layers.""",
    
    """Python is a high-level programming language known for its readability and versatility. 
    It supports multiple paradigms including object-oriented, functional, and procedural programming. 
    Python has extensive libraries for data science, web development, and automation.""",
    
    """The solar system consists of the Sun and everything that orbits it. There are eight planets: 
    Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune. The asteroid belt lies 
    between Mars and Jupiter.""",
    
    """Photosynthesis is the process by which plants convert sunlight into chemical energy. 
    Chlorophyll in plant cells absorbs light energy to convert carbon dioxide and water into 
    glucose and oxygen. This process is essential for life on Earth.""",
    
    """The transformer architecture revolutionized natural language processing. It uses self-attention 
    mechanisms to process sequences in parallel rather than sequentially. BERT, GPT, and T5 are 
    all based on transformers. Attention computes query-key-value relationships.""",
    
    """Quantum computing leverages quantum mechanical phenomena like superposition and entanglement. 
    Qubits can exist in multiple states simultaneously unlike classical bits. Quantum computers 
    could solve certain problems exponentially faster than classical computers.""",
    
    """The human immune system defends against pathogens through innate and adaptive immunity. 
    White blood cells identify and destroy foreign invaders. Vaccines train the immune system 
    to recognize specific threats without causing disease.""",
    
    """Retrieval-augmented generation combines information retrieval with text generation. 
    Instead of relying solely on parametric knowledge, RAG systems retrieve relevant documents 
    and use them as context for generating accurate answers. This reduces hallucination.""",
    
    """Database indexing improves query performance by creating data structures that allow fast lookups. 
    B-tree indexes are common in relational databases. Hash indexes provide O(1) lookups for 
    equality queries. Proper indexing can speed up queries by orders of magnitude.""",
    
    """Climate change is driven by greenhouse gas emissions, primarily from burning fossil fuels. 
    Rising temperatures cause ice caps to melt, sea levels to rise, and weather patterns to shift. 
    Renewable energy sources like solar and wind can help mitigate these effects."""
]

doc_titles = ["Machine Learning", "Python Programming", "Solar System", "Photosynthesis",
              "Transformers", "Quantum Computing", "Immune System", "RAG", "Database Indexing", "Climate Change"]

print(f"Document collection: {len(documents)} documents")
for i, (title, doc) in enumerate(zip(doc_titles, documents)):
    print(f"  [{i}] {title} ({len(doc.split())} words)")


## 2. Chunking Strategies

In [ ]:
def chunk_fixed_size(text, chunk_size=100, overlap=20):
    """Split text into fixed-size character chunks with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if len(c) > 10]

def chunk_by_sentence(text, max_sentences=2):
    """Split text into chunks of N sentences."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = ' '.join(sentences[i:i+max_sentences])
        if chunk.strip():
            chunks.append(chunk.strip())
    return chunks

# Demo both strategies on one document
print("=== Fixed-size chunking (100 chars, 20 overlap) ===")
fixed_chunks = chunk_fixed_size(documents[0], chunk_size=100, overlap=20)
for i, chunk in enumerate(fixed_chunks):
    print(f"  Chunk {i}: '{chunk[:60]}...' ({len(chunk)} chars)")

print("\n=== Sentence-based chunking (2 sentences per chunk) ===")
sent_chunks = chunk_by_sentence(documents[0], max_sentences=2)
for i, chunk in enumerate(sent_chunks):
    print(f"  Chunk {i}: '{chunk[:60]}...' ({len(chunk)} chars)")


## 3. TF-IDF Vectorization as Simple Embedding

In [ ]:
# Use sentence-based chunking for all documents
all_chunks = []
chunk_to_doc = []  # Track which document each chunk came from

for doc_idx, doc in enumerate(documents):
    chunks = chunk_by_sentence(doc, max_sentences=2)
    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_to_doc.append(doc_idx)

print(f"Total chunks: {len(all_chunks)}")
print(f"Chunks per document: {Counter(chunk_to_doc)}")

# Fit TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
chunk_vectors = vectorizer.fit_transform(all_chunks)

print(f"\nVector dimensions: {chunk_vectors.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Sample features: {list(vectorizer.vocabulary_.keys())[:15]}")


## 4. Cosine Similarity Search

In [ ]:
def search(query, vectorizer, chunk_vectors, chunks, top_k=5):
    """Search for most relevant chunks given a query."""
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, chunk_vectors).flatten()
    top_indices = similarities.argsort()[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'chunk_idx': idx,
            'score': similarities[idx],
            'text': chunks[idx],
            'doc_idx': chunk_to_doc[idx],
            'doc_title': doc_titles[chunk_to_doc[idx]]
        })
    return results

# Test search
query = "How do transformers work in NLP?"
results = search(query, vectorizer, chunk_vectors, all_chunks, top_k=5)

print(f"Query: '{query}'")
print("\nTop 5 results:")
for i, r in enumerate(results):
    print(f"  {i+1}. [{r['doc_title']}] (score: {r['score']:.4f})")
    print(f"     '{r['text'][:80]}...'")


## 5. Numpy-Based Vector Store

In [ ]:
class SimpleVectorStore:
    """A minimal vector store using numpy arrays."""
    
    def __init__(self):
        self.vectors = None
        self.documents = []
        self.metadata = []
    
    def add(self, vectors, documents, metadata=None):
        """Add vectors and associated documents."""
        if self.vectors is None:
            self.vectors = vectors.toarray() if hasattr(vectors, 'toarray') else vectors
        else:
            new_vecs = vectors.toarray() if hasattr(vectors, 'toarray') else vectors
            self.vectors = np.vstack([self.vectors, new_vecs])
        
        self.documents.extend(documents)
        if metadata:
            self.metadata.extend(metadata)
        else:
            self.metadata.extend([{}] * len(documents))
    
    def search(self, query_vector, top_k=5):
        """Find top-k most similar vectors."""
        qv = query_vector.toarray() if hasattr(query_vector, 'toarray') else query_vector
        qv = qv.flatten()
        
        # Cosine similarity
        norms = np.linalg.norm(self.vectors, axis=1)
        query_norm = np.linalg.norm(qv)
        
        # Avoid division by zero
        valid = (norms > 0) & (query_norm > 0)
        similarities = np.zeros(len(self.vectors))
        similarities[valid] = self.vectors[valid] @ qv / (norms[valid] * query_norm)
        
        top_indices = similarities.argsort()[::-1][:top_k]
        return [(self.documents[i], similarities[i], self.metadata[i]) for i in top_indices]
    
    def __len__(self):
        return len(self.documents)

# Build the vector store
store = SimpleVectorStore()
metadata = [{'doc_idx': chunk_to_doc[i], 'title': doc_titles[chunk_to_doc[i]]} for i in range(len(all_chunks))]
store.add(chunk_vectors, all_chunks, metadata)

print(f"Vector store contains {len(store)} chunks")
print(f"Vector dimensions: {store.vectors.shape[1]}")


## 6. Query the Vector Store

In [ ]:
queries = [
    "What is attention mechanism?",
    "How do plants make food?",
    "What are quantum computers?",
    "How to speed up database queries?"
]

for query in queries:
    query_vec = vectorizer.transform([query])
    results = store.search(query_vec, top_k=3)
    
    print(f"\nQuery: '{query}'")
    for i, (doc, score, meta) in enumerate(results):
        print(f"  {i+1}. [{meta['title']}] score={score:.4f}: '{doc[:60]}...'")


## 7. Assemble Context from Retrieved Documents

In [ ]:
def build_rag_prompt(query, store, vectorizer, top_k=3, max_context_chars=1000):
    """Build a RAG prompt with retrieved context."""
    query_vec = vectorizer.transform([query])
    results = store.search(query_vec, top_k=top_k)
    
    # Filter by minimum relevance score
    relevant = [(doc, score, meta) for doc, score, meta in results if score > 0.05]
    
    # Build context string
    context_parts = []
    total_chars = 0
    for doc, score, meta in relevant:
        if total_chars + len(doc) > max_context_chars:
            break
        context_parts.append(f"[Source: {meta['title']}] {doc}")
        total_chars += len(doc)
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""Answer the question based on the provided context. If the context doesn't contain 
enough information, say so.

Context:
{context}

Question: {query}

Answer:"""
    return prompt, relevant

# Demo the full RAG pipeline
query = "How does retrieval-augmented generation reduce hallucination?"
prompt, sources = build_rag_prompt(query, store, vectorizer)

print("=" * 70)
print("FULL RAG PROMPT")
print("=" * 70)
print(prompt)
print("\n" + "=" * 70)
print(f"Sources used: {len(sources)}")
for doc, score, meta in sources:
    print(f"  - {meta['title']} (relevance: {score:.4f})")


## 8. Full RAG Pipeline Visualization

In [ ]:
def visualize_rag_pipeline(query, store, vectorizer):
    """Visualize the RAG pipeline steps."""
    print("RAG Pipeline Execution")
    print("=" * 60)
    
    # Step 1: Query
    print(f"\n1. QUERY: '{query}'")
    
    # Step 2: Embed
    query_vec = vectorizer.transform([query])
    non_zero = query_vec.nonzero()[1]
    feature_names = vectorizer.get_feature_names_out()
    print(f"\n2. EMBED: Query vector has {len(non_zero)} non-zero features")
    top_features = sorted([(feature_names[i], query_vec[0, i]) for i in non_zero], 
                          key=lambda x: -x[1])[:5]
    print(f"   Top terms: {[(f, f'{s:.3f}') for f, s in top_features]}")
    
    # Step 3: Search
    results = store.search(query_vec, top_k=3)
    print(f"\n3. SEARCH: Found top-3 chunks")
    for i, (doc, score, meta) in enumerate(results):
        print(f"   [{score:.4f}] {meta['title']}: '{doc[:50]}...'")
    
    # Step 4: Context assembly
    context = "\n".join([doc for doc, score, meta in results if score > 0.05])
    print(f"\n4. CONTEXT: Assembled {len(context)} characters from {sum(1 for _,s,_ in results if s>0.05)} sources")
    
    # Step 5: Prompt
    print(f"\n5. PROMPT: Ready for LLM (would include context + query)")
    print(f"   Total prompt length: ~{len(context) + len(query) + 100} characters")

visualize_rag_pipeline("What is self-attention in transformers?", store, vectorizer)


## 9. Evaluate Retrieval Quality

In [ ]:
# Ground truth: which documents are relevant for each query
eval_queries = [
    {"query": "machine learning and neural networks", "relevant_docs": [0, 4]},
    {"query": "how do plants convert sunlight to energy", "relevant_docs": [3]},
    {"query": "transformer attention mechanism NLP", "relevant_docs": [4, 7]},
    {"query": "quantum superposition and qubits", "relevant_docs": [5]},
    {"query": "index data structures for fast queries", "relevant_docs": [8]},
]

def evaluate_retrieval(queries, store, vectorizer, k_values=[1, 3, 5]):
    """Compute Recall@K and Precision@K."""
    results = {k: {'precision': [], 'recall': []} for k in k_values}
    
    for q_info in queries:
        query_vec = vectorizer.transform([q_info['query']])
        search_results = store.search(query_vec, top_k=max(k_values))
        
        retrieved_docs = [meta['doc_idx'] for _, _, meta in search_results]
        relevant = set(q_info['relevant_docs'])
        
        for k in k_values:
            retrieved_at_k = set(retrieved_docs[:k])
            hits = len(retrieved_at_k & relevant)
            precision = hits / k
            recall = hits / len(relevant) if relevant else 0
            results[k]['precision'].append(precision)
            results[k]['recall'].append(recall)
    
    return results

eval_results = evaluate_retrieval(eval_queries, store, vectorizer)

print("Retrieval Evaluation")
print("=" * 40)
for k, metrics in eval_results.items():
    avg_p = np.mean(metrics['precision'])
    avg_r = np.mean(metrics['recall'])
    print(f"  @{k}: Precision={avg_p:.3f}, Recall={avg_r:.3f}")


## 10. Compare Chunk Sizes

In [ ]:
chunk_configs = [
    ('1 sentence', 1),
    ('2 sentences', 2),
    ('3 sentences', 3),
    ('Full document', 10),
]

config_results = []

for name, n_sent in chunk_configs:
    # Re-chunk
    chunks = []
    c2d = []
    for doc_idx, doc in enumerate(documents):
        doc_chunks = chunk_by_sentence(doc, max_sentences=n_sent)
        chunks.extend(doc_chunks)
        c2d.extend([doc_idx] * len(doc_chunks))
    
    # Re-vectorize
    vec = TfidfVectorizer(stop_words='english', max_features=500)
    vecs = vec.fit_transform(chunks)
    
    # Build store
    s = SimpleVectorStore()
    meta = [{'doc_idx': c2d[i], 'title': doc_titles[c2d[i]]} for i in range(len(chunks))]
    s.add(vecs, chunks, meta)
    
    # Evaluate
    res = evaluate_retrieval(eval_queries, s, vec, k_values=[1, 3, 5])
    config_results.append({
        'config': name,
        'n_chunks': len(chunks),
        'recall@3': np.mean(res[3]['recall']),
        'precision@3': np.mean(res[3]['precision']),
    })

df = pd.DataFrame(config_results)
print(df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(df))
ax.bar([i-0.15 for i in x], df['recall@3'], 0.3, label='Recall@3', color='steelblue')
ax.bar([i+0.15 for i in x], df['precision@3'], 0.3, label='Precision@3', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(df['config'])
ax.set_ylabel('Score')
ax.set_title('Effect of Chunk Size on Retrieval Quality')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


## 11. Hybrid Search: TF-IDF + BM25-like Scoring

In [ ]:
def bm25_score(query, documents, k1=1.5, b=0.75):
    """Compute BM25 scores for a query against documents."""
    # Tokenize
    query_terms = query.lower().split()
    doc_tokens = [doc.lower().split() for doc in documents]
    
    # Document lengths
    doc_lens = np.array([len(d) for d in doc_tokens])
    avgdl = doc_lens.mean()
    N = len(documents)
    
    scores = np.zeros(N)
    
    for term in query_terms:
        # Document frequency
        df = sum(1 for doc in doc_tokens if term in doc)
        if df == 0:
            continue
        
        # IDF component
        idf = np.log((N - df + 0.5) / (df + 0.5) + 1)
        
        for i, doc in enumerate(doc_tokens):
            # Term frequency in document
            tf = doc.count(term)
            # BM25 formula
            tf_norm = (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * doc_lens[i] / avgdl))
            scores[i] += idf * tf_norm
    
    return scores

def hybrid_search(query, chunks, vectorizer, chunk_vectors, alpha=0.5, top_k=5):
    """Combine TF-IDF cosine similarity with BM25 scoring."""
    # TF-IDF scores
    query_vec = vectorizer.transform([query])
    tfidf_scores = cosine_similarity(query_vec, chunk_vectors).flatten()
    
    # BM25 scores
    bm25_scores = bm25_score(query, chunks)
    
    # Normalize both to [0, 1]
    if tfidf_scores.max() > 0:
        tfidf_norm = tfidf_scores / tfidf_scores.max()
    else:
        tfidf_norm = tfidf_scores
    
    if bm25_scores.max() > 0:
        bm25_norm = bm25_scores / bm25_scores.max()
    else:
        bm25_norm = bm25_scores
    
    # Hybrid score
    hybrid_scores = alpha * tfidf_norm + (1 - alpha) * bm25_norm
    
    top_indices = hybrid_scores.argsort()[::-1][:top_k]
    return [(chunks[i], hybrid_scores[i], tfidf_norm[i], bm25_norm[i]) for i in top_indices]

# Compare search methods
query = "How does attention work in neural networks?"
print(f"Query: '{query}'\n")

hybrid_results = hybrid_search(query, all_chunks, vectorizer, chunk_vectors, alpha=0.5, top_k=5)

print(f"{'Rank':<5} {'Hybrid':<8} {'TF-IDF':<8} {'BM25':<8} {'Text':<50}")
print("-" * 80)
for i, (text, hybrid, tfidf, bm25) in enumerate(hybrid_results):
    print(f"{i+1:<5} {hybrid:<8.4f} {tfidf:<8.4f} {bm25:<8.4f} {text[:50]}...")


In [ ]:
# Compare alpha values for hybrid search
alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
alpha_labels = ['BM25 only', '25/75', '50/50', '75/25', 'TF-IDF only']

print("Effect of alpha on hybrid search ranking")
print("(alpha=1 means TF-IDF only, alpha=0 means BM25 only)\n")

test_query = "transformer self-attention mechanism"
print(f"Query: '{test_query}'\n")

for alpha, label in zip(alphas, alpha_labels):
    results = hybrid_search(test_query, all_chunks, vectorizer, chunk_vectors, alpha=alpha, top_k=3)
    top_doc = results[0][0][:50]
    print(f"  alpha={alpha:.2f} ({label:>10}): top result = '{top_doc}...'")


## 12. Retrieval Quality Visualization

In [ ]:
# Visualize similarity heatmap for all queries vs all chunks
test_queries = ["machine learning models", "plant biology", "quantum physics", 
                "programming languages", "climate and environment"]

query_vecs = vectorizer.transform(test_queries)
sim_matrix = cosine_similarity(query_vecs, chunk_vectors)

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(sim_matrix, aspect='auto', cmap='YlOrRd')
ax.set_yticks(range(len(test_queries)))
ax.set_yticklabels(test_queries, fontsize=9)
ax.set_xlabel('Chunk Index')
ax.set_title('Query-Chunk Similarity Heatmap')
plt.colorbar(im, ax=ax, label='Cosine Similarity')
plt.tight_layout()
plt.show()


## Summary

Key components of a RAG system:
1. **Document chunking** - Split documents into retrievable units
2. **Embedding/Vectorization** - Convert text to dense vectors (here: TF-IDF)
3. **Vector store** - Index and search vectors efficiently
4. **Retrieval** - Find top-k relevant chunks for a query
5. **Context assembly** - Build a prompt with retrieved context
6. **Generation** - LLM generates answer grounded in context

In production, you'd use dense embeddings (e.g., sentence-transformers) and vector databases (FAISS, Pinecone, Weaviate).

In [ ]:
print("RAG Pipeline Summary")
print("=" * 50)
print(f"Documents: {len(documents)}")
print(f"Chunks: {len(all_chunks)}")
print(f"Vector dimensions: {chunk_vectors.shape[1]}")
print(f"Avg Recall@3: {np.mean(eval_results[3]['recall']):.3f}")
print(f"Avg Precision@3: {np.mean(eval_results[3]['precision']):.3f}")
